# LeetCode 1352 — Product of the Last K Numbers bằng Python

Link đề: https://leetcode.com/problems/product-of-the-last-k-numbers/ · Độ khó: **Medium** · Chủ đề: Array, Math, Design, Data Stream, Prefix Sum

Skeleton LeetCode (bài **thiết kế lớp**):

```python
class ProductOfNumbers:

    def __init__(self):
        ...

    def add(self, num: int) -> None:
        ...

    def getProduct(self, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Làm quen bài **data stream** (dòng dữ liệu): phần tử đến dần, truy vấn xen kẽ.
2. Hiểu **tích tiền tố (prefix product)** — "anh em" của tổng tiền tố: tích đoạn = `P[r] / P[l]`.
3. Xử lý **số 0** — điểm khiến phép chia không dùng được: **reset** mảng tiền tố khi gặp 0, hoặc lưu **vị trí số 0 cuối cùng**.
4. Đi từ brute force `O(k)` mỗi truy vấn → `O(1)` cho cả `add` và `getProduct` (đúng follow-up của đề).
5. Viết bộ **giả lập chuỗi lệnh** để kiểm thử lớp, đối chiếu ngẫu nhiên với brute force.

## 2. Đề bài & ví dụ

Thiết kế cấu trúc nhận một dòng số nguyên và trả về **tích của `k` số cuối cùng**.

- `ProductOfNumbers()` khởi tạo với dòng rỗng.
- `add(num)` thêm `num` vào cuối dòng.
- `getProduct(k)` trả về tích của `k` số cuối. Đảm bảo dòng luôn có ít nhất `k` số.

Đề đảm bảo: tích của **mọi đoạn liên tiếp** luôn vừa số nguyên 32-bit.

**Ví dụ:**
```
["ProductOfNumbers","add","add","add","add","add","getProduct","getProduct","getProduct","add","getProduct"]
[[],[3],[0],[2],[5],[4],[2],[3],[4],[8],[2]]
Output: [null,null,null,null,null,null,20,40,0,null,32]
```
Dòng sau 5 lệnh `add`: `[3,0,2,5,4]`. `getProduct(2) = 5·4 = 20`, `getProduct(3) = 2·5·4 = 40`, `getProduct(4) = 0·2·5·4 = 0`. Thêm 8: `[3,0,2,5,4,8]`, `getProduct(2) = 4·8 = 32`.

**Ràng buộc:** `0 <= num <= 100`; `1 <= k <= 4·10^4`; tối đa `4·10^4` lời gọi.

**Follow-up:** làm cả `add` và `getProduct` trong `O(1)`.

In [1]:
import random
import time
from math import prod

EX_OPS = ["ProductOfNumbers", "add", "add", "add", "add", "add", "getProduct", "getProduct", "getProduct", "add", "getProduct"]
EX_ARGS = [[], [3], [0], [2], [5], [4], [2], [3], [4], [8], [2]]
EX_OUT = [None, None, None, None, None, None, 20, 40, 0, None, 32]

def run(cls, ops, args):
    """Bộ GIẢ LẬP kiểu LeetCode: tạo đối tượng rồi gọi lần lượt các phương thức."""
    obj = cls(*args[0])
    out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))
    return out

print("Output mong đợi:", EX_OUT)

Output mong đợi: [None, None, None, None, None, None, 20, 40, 0, None, 32]


## 3. Kiến thức nền

### 3.1 Bài design & bộ giả lập

LeetCode tạo một đối tượng rồi gọi các phương thức theo thứ tự trong `ops`/`args`. Hàm `run` ở trên mô phỏng điều đó bằng `getattr(obj, "add")(*[3])`. Mọi cài đặt trong notebook đều được kiểm thử qua `run`.

In [2]:
class Echo:
    def __init__(self):
        self.log = []
    def add(self, x):
        self.log.append(x)
    def getProduct(self, k):
        return self.log[-k:]

print(run(Echo, ["Echo", "add", "add", "getProduct"], [[], [7], [9], [2]]))

[None, None, None, [7, 9]]


### 3.2 Cắt lát `a[-k:]` và `math.prod`

`a[-k:]` lấy `k` phần tử cuối; `math.prod(iterable)` tính tích (mặc định tích rỗng = 1).

In [3]:
a = [3, 0, 2, 5, 4]
print(a[-2:], prod(a[-2:]))
print(a[-4:], prod(a[-4:]))
print("tích rỗng:", prod([]))

[5, 4] 20
[0, 2, 5, 4] 0
tích rỗng: 1


### 3.3 Tổng tiền tố → tích tiền tố

- Tổng tiền tố: `S[0] = 0`, `S[i] = S[i−1] + a[i−1]` ⇒ tổng `k` số cuối `= S[n] − S[n−k]`.
- Tích tiền tố: `P[0] = 1`, `P[i] = P[i−1] · a[i−1]` ⇒ tích `k` số cuối `= P[n] / P[n−k]` — **chỉ khi không có số 0**!

In [4]:
a = [2, 5, 4, 8]
P = [1]
for x in a:
    P.append(P[-1] * x)
print("P =", P)
k = 2
print(f"tích {k} số cuối = P[{len(a)}] // P[{len(a) - k}] =", P[len(a)] // P[len(a) - k], "| kiểm tra:", prod(a[-k:]))

P = [1, 2, 10, 40, 320]
tích 2 số cuối = P[4] // P[2] = 32 | kiểm tra: 32


### 3.4 Vì sao số 0 phá hỏng phép chia

Khi một số 0 xuất hiện, mọi `P[i]` từ đó trở đi đều bằng 0 ⇒ `P[n] / P[n−k]` thành `0 / 0` ⇒ `ZeroDivisionError`. Ta cần một cách "quên" phần trước số 0.

In [5]:
a = [3, 0, 2, 5, 4]
P = [1]
for x in a:
    P.append(P[-1] * x)
print("P =", P)
try:
    print(P[5] // P[3])
except ZeroDivisionError as e:
    print("Lỗi:", type(e).__name__, "-", e)

P = [1, 3, 0, 0, 0, 0]
Lỗi: ZeroDivisionError - integer division or modulo by zero


### 3.5 Chia nguyên `//` và chia thực `/`

Tích là số nguyên nên dùng `//` (chia hết, không sai số). `/` trả về `float` — với số lớn có thể mất chính xác, và LeetCode mong đợi `int`.

In [6]:
x, y = 2**60 * 3, 2**60
print(x // y, x / y, type(x / y).__name__)
big = 10**17 + 1
print("float mất chính xác:", int(big * 7 / 7) == big)

3 3.0 float
float mất chính xác: False


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ với ý tưởng "reset khi gặp 0". Ta chỉ giữ tích tiền tố **từ sau số 0 gần nhất**:

| Lệnh | dòng | `pre` (tích tiền tố từ sau số 0 cuối) | Ghi chú |
|---|---|---|---|
| add 3 | [3] | [1, 3] | |
| add 0 | [3,0] | [1] | gặp 0 ⇒ reset |
| add 2 | [3,0,2] | [1, 2] | |
| add 5 | [3,0,2,5] | [1, 2, 10] | |
| add 4 | [3,0,2,5,4] | [1, 2, 10, 40] | |
| get 2 | | `40 // pre[-3] = 40 // 2 = 20` | |
| get 3 | | `40 // pre[-4] = 40 // 1 = 40` | |
| get 4 | | `k = 4 ≥ len(pre) = 4` ⇒ **0** | đoạn chứa số 0 |
| add 8 | | [1, 2, 10, 40, 320] | |
| get 2 | | `320 // 10 = 32` | |

**Trực giác:** `len(pre) − 1` = số phần tử **từ sau số 0 cuối**. Nếu `k` vượt quá con số đó, đoạn `k` số cuối chắc chắn chứa số 0 ⇒ tích bằng 0.

In [7]:
pre = [1]
stream = []
for op, a in zip(EX_OPS[1:], EX_ARGS[1:]):
    if op == "add":
        stream.append(a[0])
        pre = [1] if a[0] == 0 else pre + [pre[-1] * a[0]]
        print(f"add {a[0]:>2}: dòng={stream}, pre={pre}")
    else:
        k = a[0]
        ans = 0 if k >= len(pre) else pre[-1] // pre[-k - 1]
        print(f"get {k}: -> {ans}")

add  3: dòng=[3], pre=[1, 3]
add  0: dòng=[3, 0], pre=[1]
add  2: dòng=[3, 0, 2], pre=[1, 2]
add  5: dòng=[3, 0, 2, 5], pre=[1, 2, 10]
add  4: dòng=[3, 0, 2, 5, 4], pre=[1, 2, 10, 40]
get 2: -> 20
get 3: -> 40
get 4: -> 0
add  8: dòng=[3, 0, 2, 5, 4, 8], pre=[1, 2, 10, 40, 320]
get 2: -> 32


## 5. Cách 1 — Brute Force / baseline: nhân trực tiếp `k` số cuối

**Ý tưởng:** lưu toàn bộ dòng trong list; `getProduct(k)` nhân `k` phần tử cuối.

In [8]:
class ProductBrute:
    def __init__(self):
        self.a = []

    def add(self, num):
        self.a.append(num)

    def getProduct(self, k):
        p = 1
        for x in self.a[-k:]:
            p *= x
        return p

got = run(ProductBrute, EX_OPS, EX_ARGS)
print(got, "| đúng?", got == EX_OUT)

[None, None, None, None, None, None, 20, 40, 0, None, 32] | đúng? True


**Độ phức tạp:** `add` `O(1)`; `getProduct` `O(k)`. Xấu nhất `2·10^4` lần `add` rồi `2·10^4` lần `getProduct(2·10^4)` ⇒ `4·10^8` phép nhân — quá chậm trong Python.

Cải tiến nhỏ (Cách 1b): duyệt **ngược** từ cuối và **dừng ngay khi gặp 0** — nhanh hơn khi có nhiều số 0 nhưng worst-case vẫn `O(k)`.

In [9]:
class ProductBruteEarlyStop(ProductBrute):
    def getProduct(self, k):
        p = 1
        a = self.a
        for i in range(len(a) - 1, len(a) - k - 1, -1):
            if a[i] == 0:
                return 0
            p *= a[i]
        return p

print(run(ProductBruteEarlyStop, EX_OPS, EX_ARGS) == EX_OUT)

True


**Nút thắt:** mỗi truy vấn nhân lại từ đầu những số đã nhân ở truy vấn trước. Giống bài "tổng `k` số cuối" — lời giải chuẩn là **tiền tố**; chỉ có số 0 là trở ngại.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tích tiền tố):** nếu đoạn `k` số cuối **không chứa 0** thì tích `= P[n] / P[n−k]`.

**Insight 2 (xử lý 0):** mọi đoạn chứa số 0 có tích 0. Đoạn `k` số cuối chứa 0 ⇔ `k` lớn hơn số phần tử nằm **sau số 0 cuối cùng**. Vì vậy ta chỉ cần tích tiền tố của phần **sau số 0 cuối** ⇒ khi gặp 0, **reset** `pre = [1]`.

**Bất biến:** `pre[i]` = tích của `i` phần tử đầu tiên **sau số 0 gần nhất** (hoặc từ đầu dòng nếu chưa có 0); `len(pre) − 1 = m` là số phần tử đó.

**Quy tắc nếu… thì…:**
- Nếu `num == 0` thì `pre = [1]`; ngược lại `pre.append(pre[-1] * num)`.
- Nếu `k > m` (tức `k ≥ len(pre)`) thì trả `0`; ngược lại trả `pre[-1] // pre[-k-1]`.

Bonus: nhờ reset, các tích luôn nhỏ (≤ giới hạn 32-bit theo đề) — không có số nguyên khổng lồ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tích tiền tố + reset khi gặp 0

**Giải thích từng đoạn:**
1. `self.pre = [1]` — phần tử canh (sentinel) `1` giúp công thức `pre[-1] // pre[-k-1]` đúng cả khi `k = m`.
2. `add`: số 0 ⇒ reset; khác 0 ⇒ nối tích mới.
3. `getProduct`: so `k` với `len(pre)`.

In [10]:
class ProductPrefixReset:
    def __init__(self):
        self.pre = [1]

    def add(self, num):
        if num == 0:
            self.pre = [1]
        else:
            self.pre.append(self.pre[-1] * num)

    def getProduct(self, k):
        if k >= len(self.pre):
            return 0
        return self.pre[-1] // self.pre[-k - 1]

print(run(ProductPrefixReset, EX_OPS, EX_ARGS))

[None, None, None, None, None, None, 20, 40, 0, None, 32]


Code nộp LeetCode (Cách 2):

In [11]:
class ProductOfNumbers:

    def __init__(self):
        self.pre = [1]

    def add(self, num: int) -> None:
        if num == 0:
            self.pre = [1]
        else:
            self.pre.append(self.pre[-1] * num)

    def getProduct(self, k: int) -> int:
        if k >= len(self.pre):
            return 0
        return self.pre[-1] // self.pre[-k - 1]

print(run(ProductOfNumbers, EX_OPS, EX_ARGS) == EX_OUT)

True


**Độ phức tạp:** `add` `O(1)` (khấu hao — `append`), `getProduct` `O(1)`. Bộ nhớ `O(n)`.

### 7.2 Cách 3 — Giữ nguyên độ dài, lưu vị trí số 0 cuối cùng

**Giải thích:** không xóa mảng mà vẫn `append` cho mọi số; khi gặp 0 thì ghi `pre.append(1)` (tích "khởi động lại") và nhớ `last_zero` = vị trí (1-based) của số 0 đó. Truy vấn: đoạn `k` số cuối là các vị trí `n−k+1..n`; nếu `last_zero ≥ n−k+1` thì chứa 0 ⇒ trả 0; ngược lại `pre[n] // pre[n−k]`.

Ưu điểm: `pre` có chỉ số trùng với vị trí trong dòng — dễ mở rộng sang truy vấn đoạn bất kỳ.

In [12]:
class ProductLastZero:
    def __init__(self):
        self.pre = [1]          # pre[i]: tích từ sau số 0 cuối (trước/tại i) đến i
        self.last_zero = 0      # vị trí 1-based của số 0 cuối, 0 nếu chưa có

    def add(self, num):
        if num == 0:
            self.pre.append(1)
            self.last_zero = len(self.pre) - 1
        else:
            self.pre.append(self.pre[-1] * num)

    def getProduct(self, k):
        n = len(self.pre) - 1
        if self.last_zero >= n - k + 1:
            return 0
        return self.pre[n] // self.pre[n - k]

print(run(ProductLastZero, EX_OPS, EX_ARGS) == EX_OUT)

True


Code nộp LeetCode (Cách 3):

In [13]:
class ProductOfNumbers:

    def __init__(self):
        self.pre = [1]
        self.last_zero = 0

    def add(self, num: int) -> None:
        if num == 0:
            self.pre.append(1)
            self.last_zero = len(self.pre) - 1
        else:
            self.pre.append(self.pre[-1] * num)

    def getProduct(self, k: int) -> int:
        n = len(self.pre) - 1
        if self.last_zero >= n - k + 1:
            return 0
        return self.pre[n] // self.pre[n - k]

print(run(ProductOfNumbers, EX_OPS, EX_ARGS))

[None, None, None, None, None, None, 20, 40, 0, None, 32]


**Độ phức tạp:** như Cách 2, `O(1)` / `O(1)`, bộ nhớ `O(n)` (không giải phóng phần trước số 0).

### 7.3 Cách 4 — Đếm số 0 bằng tiền tố + tích các số khác 0

**Giải thích:** hai mảng tiền tố song song: `Z[i]` = số lượng số 0 trong `i` phần tử đầu; `Q[i]` = tích các số **khác 0** trong `i` phần tử đầu (số 0 coi như 1). Đoạn `(n−k, n]` có tích 0 nếu `Z[n] − Z[n−k] > 0`; ngược lại `Q[n] // Q[n−k]`.

Cách này tổng quát nhất (trả lời được **mọi đoạn** `[l, r]`), nhưng `Q` có thể thành số rất lớn vì không reset (Python vẫn xử lý được nhờ số nguyên không giới hạn, song chậm hơn). Ở bài này ta "chuẩn hóa" bằng cách reset `Q` sau số 0 như Cách 3 — khi đó nó trùng Cách 3. Ta cài bản không reset để minh họa.

In [14]:
class ProductZeroCount:
    def __init__(self):
        self.Z = [0]
        self.Q = [1]

    def add(self, num):
        self.Z.append(self.Z[-1] + (num == 0))
        self.Q.append(self.Q[-1] * (num if num else 1))

    def getProduct(self, k):
        n = len(self.Z) - 1
        if self.Z[n] - self.Z[n - k] > 0:
            return 0
        return self.Q[n] // self.Q[n - k]

print(run(ProductZeroCount, EX_OPS, EX_ARGS) == EX_OUT)

True


## 8. TRACE từng bước

Trace Cách 2 trên ví dụ đề, in `pre` sau mỗi lệnh và phép chia dùng cho mỗi truy vấn.

In [15]:
class ProductTrace(ProductPrefixReset):
    def add(self, num):
        super().add(num)
        note = "  (reset vì gặp 0)" if num == 0 else ""
        print(f"add({num:>2})        pre = {self.pre}{note}")

    def getProduct(self, k):
        if k >= len(self.pre):
            print(f"getProduct({k}): k={k} >= len(pre)={len(self.pre)} -> đoạn chứa 0 -> 0")
            return 0
        a, b = self.pre[-1], self.pre[-k - 1]
        print(f"getProduct({k}): pre[-1]={a} // pre[-{k + 1}]={b} = {a // b}")
        return a // b

def trace_run(ops, args):
    return run(ProductTrace, ops, args)

print("Kết quả:", trace_run(EX_OPS, EX_ARGS))

add( 3)        pre = [1, 3]
add( 0)        pre = [1]  (reset vì gặp 0)
add( 2)        pre = [1, 2]
add( 5)        pre = [1, 2, 10]
add( 4)        pre = [1, 2, 10, 40]
getProduct(2): pre[-1]=40 // pre[-3]=2 = 20
getProduct(3): pre[-1]=40 // pre[-4]=1 = 40
getProduct(4): k=4 >= len(pre)=4 -> đoạn chứa 0 -> 0
add( 8)        pre = [1, 2, 10, 40, 320]
getProduct(2): pre[-1]=320 // pre[-3]=10 = 32
Kết quả: [None, None, None, None, None, None, 20, 40, 0, None, 32]


Bảng trạng thái (Cách 3 — giữ nguyên độ dài):

| Lệnh | `pre` | `last_zero` | n | Truy vấn |
|---|---|---|---|---|
| add 3 | [1,3] | 0 | 1 | |
| add 0 | [1,3,1] | 2 | 2 | |
| add 2 | [1,3,1,2] | 2 | 3 | |
| add 5 | [1,3,1,2,10] | 2 | 4 | |
| add 4 | [1,3,1,2,10,40] | 2 | 5 | |
| get 2 | | | 5 | `2 ≥ 4`? không ⇒ `40 // pre[3]=2` = 20 |
| get 3 | | | 5 | `2 ≥ 3`? không ⇒ `40 // pre[2]=1` = 40 |
| get 4 | | | 5 | `2 ≥ 2`? **có** ⇒ 0 |
| add 8 | [...,40,320] | 2 | 6 | |
| get 2 | | | 6 | `2 ≥ 5`? không ⇒ `320 // pre[4]=10` = 32 |

In [16]:
class ProductLastZeroTrace(ProductLastZero):
    def getProduct(self, k):
        n = len(self.pre) - 1
        cond = self.last_zero >= n - k + 1
        r = super().getProduct(k)
        print(f"get {k}: n={n}, last_zero={self.last_zero}, đoạn=[{n - k + 1},{n}], chứa 0? {cond} -> {r}")
        return r

run(ProductLastZeroTrace, EX_OPS, EX_ARGS);

get 2: n=5, last_zero=2, đoạn=[4,5], chứa 0? False -> 20
get 3: n=5, last_zero=2, đoạn=[3,5], chứa 0? False -> 40
get 4: n=5, last_zero=2, đoạn=[2,5], chứa 0? True -> 0
get 2: n=6, last_zero=2, đoạn=[5,6], chứa 0? False -> 32


## 9. Code nộp LeetCode

In [17]:
class ProductOfNumbers:

    def __init__(self):
        self.pre = [1]                 # tích tiền tố tính từ sau số 0 gần nhất

    def add(self, num: int) -> None:
        if num == 0:
            self.pre = [1]             # mọi đoạn đi qua số 0 có tích 0 -> quên phần trước
        else:
            self.pre.append(self.pre[-1] * num)

    def getProduct(self, k: int) -> int:
        if k >= len(self.pre):         # k số cuối vượt qua số 0 gần nhất
            return 0
        return self.pre[-1] // self.pre[-k - 1]

got = run(ProductOfNumbers, EX_OPS, EX_ARGS)
assert got == EX_OUT
print(got)

[None, None, None, None, None, None, 20, 40, 0, None, 32]


## 10. Vì sao thuật toán đúng

Gọi dòng hiện tại là `a_1..a_n`, `z` là vị trí số 0 cuối cùng (`z = 0` nếu chưa có), `m = n − z`.

**Bất biến:** `pre` có độ dài `m + 1` và `pre[i] = a_{z+1}·…·a_{z+i}` (với `pre[0] = 1`).
- Khởi tạo: `n = 0, z = 0, m = 0`, `pre = [1]` ✔.
- `add(0)`: `z ← n+1`, `m ← 0`, `pre ← [1]` ✔.
- `add(x ≠ 0)`: `m ← m+1`, nối `pre[m] = pre[m−1]·x` ✔.

**Truy vấn `k`:** đoạn cuối là `a_{n−k+1}..a_n`.
- Nếu `k > m` (⇔ `k ≥ len(pre)`): `n − k + 1 ≤ z` ⇒ đoạn chứa `a_z = 0` ⇒ tích 0 ✔.
- Nếu `k ≤ m`: đoạn nằm hoàn toàn sau `z`, không có số 0, và
  `pre[m] / pre[m−k] = (a_{z+1}…a_n) / (a_{z+1}…a_{n−k}) = a_{n−k+1}…a_n`. Phép chia là chia hết (thương là số nguyên), và `pre[-k-1] = pre[m−k] ≠ 0` ✔.

## 11. Phân tích Time / Space

Gọi `n` = số lệnh `add`, `k` = tham số truy vấn.

| Cách | add | getProduct | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | `O(1)` | `O(k)` | `O(n)` | |
| 1b. Brute + dừng tại 0 | `O(1)` | `O(min(k, khoảng cách tới số 0))` | `O(n)` | worst vẫn `O(k)` |
| 2. Tiền tố + reset | `O(1)` khấu hao | `O(1)` | `O(n)` (thực tế chỉ phần sau số 0) | lời giải chuẩn |
| 3. Tiền tố + last_zero | `O(1)` khấu hao | `O(1)` | `O(n)` | chỉ số trùng vị trí dòng |
| 4. Đếm 0 + tích khác 0 | `O(1)` khấu hao* | `O(1)`* | `O(n)` | *số nguyên lớn dần ⇒ phép nhân/chia chậm hơn |

`O(1)` khấu hao: `list.append` thỉnh thoảng phải cấp phát lại, nhưng trung bình mỗi lần là hằng số.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
IMPLS = [ProductBrute, ProductBruteEarlyStop, ProductPrefixReset, ProductLastZero, ProductZeroCount, ProductOfNumbers]

cases = [
    (EX_OPS, EX_ARGS, EX_OUT),
    # chỉ một số
    (["P", "add", "getProduct"], [[], [7], [1]], [None, None, 7]),
    # số 0 là phần tử mới nhất
    (["P", "add", "add", "getProduct", "getProduct"], [[], [5], [0], [1], [2]], [None, None, None, 0, 0]),
    # số 0 ở đầu, k đúng bằng số phần tử sau 0, rồi k vượt qua
    (["P", "add", "add", "add", "getProduct", "getProduct"], [[], [0], [2], [3], [2], [3]], [None, None, None, None, 6, 0]),
    # nhiều số 1
    (["P"] + ["add"] * 4 + ["getProduct"], [[]] + [[1]] * 4 + [[4]], [None] * 5 + [1]),
    # hai số 0 liên tiếp
    (["P", "add", "add", "add", "getProduct", "getProduct"], [[], [0], [0], [9], [1], [2]], [None, None, None, None, 9, 0]),
]
for ops, args, want in cases:
    for cls in IMPLS:
        got = run(cls, ops, args)
        assert got == want, (cls.__name__, ops, args, got, want)
print(f"{len(cases)} kịch bản × {len(IMPLS)} cài đặt: đúng hết")

6 kịch bản × 6 cài đặt: đúng hết


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh dòng số ngẫu nhiên với nhiều số 0, 1, 2 để tích không quá lớn (đúng tinh thần "vừa 32-bit"); `k` luôn ≤ số phần tử hiện có.

In [19]:
def random_ops(rng, n_calls):
    ops, args, size = ["P"], [[]], 0
    for _ in range(n_calls):
        if size == 0 or rng.random() < 0.55:
            ops.append("add")
            args.append([rng.choice([0, 1, 1, 1, 2, 2, 3, 5, 0, 1])])
            size += 1
        else:
            ops.append("getProduct")
            args.append([rng.randint(1, size)])
    return ops, args

rng = random.Random(1352)
for _ in range(400):
    ops, args = random_ops(rng, rng.randint(1, 60))
    ref = run(ProductBrute, ops, args)
    for cls in IMPLS[1:]:
        assert run(cls, ops, args) == ref, cls.__name__
print("400 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force")
# nhờ reset, các giá trị trong pre luôn là tích của một đoạn liên tiếp -> nhỏ
ops, args = random_ops(random.Random(3), 2000)
obj = ProductPrefixReset()
biggest = 1
for op, a in zip(ops[1:], args[1:]):
    if op == "add":
        obj.add(*a)
        biggest = max(biggest, obj.pre[-1])
print("Giá trị lớn nhất từng xuất hiện trong pre (Cách 2):", biggest)

400 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force
Giá trị lớn nhất từng xuất hiện trong pre (Cách 2): 3888000


### 12.3 Hiệu năng ở giới hạn

`2·10^4` lần `add` (toàn số 1 và vài số 2 để tích nhỏ) rồi `2·10^4` lần `getProduct(k)` với `k` lớn. So brute force với Cách 2.

In [20]:
rng = random.Random(9)
N = 2 * 10**4
ops = ["P"] + ["add"] * N + ["getProduct"] * N
vals = [1] * N
for i in rng.sample(range(N), 20):
    vals[i] = 2
args = [[]] + [[v] for v in vals] + [[rng.randint(N // 2, N)] for _ in range(N)]
res = {}
for cls in [ProductPrefixReset, ProductLastZero]:
    st = time.perf_counter()
    res[cls.__name__] = run(cls, ops, args)
    print(f"{cls.__name__:22} {time.perf_counter() - st:.3f}s cho {N} add + {N} getProduct")
assert res["ProductPrefixReset"] == res["ProductLastZero"]
Q = 1000                                   # brute force chỉ chạy 1000 truy vấn (đủ thấy chậm)
st = time.perf_counter()
brute = run(ProductBruteEarlyStop, ops[:N + 1 + Q], args[:N + 1 + Q])
el = time.perf_counter() - st
print(f"ProductBruteEarlyStop  {el:.3f}s cho {N} add + chỉ {Q} getProduct (ước tính {el * N / Q:.1f}s cho {N})")
assert brute == res["ProductPrefixReset"][:N + 1 + Q]
print("Kết quả trùng nhau; brute force chậm hơn hàng trăm lần vì O(k) mỗi truy vấn")

ProductPrefixReset     0.010s cho 20000 add + 20000 getProduct
ProductLastZero        0.009s cho 20000 add + 20000 getProduct


ProductBruteEarlyStop  0.796s cho 20000 add + chỉ 1000 getProduct (ước tính 15.9s cho 20000)
Kết quả trùng nhau; brute force chậm hơn hàng trăm lần vì O(k) mỗi truy vấn


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không xử lý số 0 (chia cho 0)

In [21]:
class WrongNoZero:
    def __init__(self):
        self.pre = [1]
    def add(self, num):
        self.pre.append(self.pre[-1] * num)
    def getProduct(self, k):
        return self.pre[-1] // self.pre[-k - 1]

try:
    print(run(WrongNoZero, EX_OPS, EX_ARGS))
except ZeroDivisionError as e:
    print("Chạy ví dụ đề -> ZeroDivisionError:", e)
print("đúng:", EX_OUT)

Chạy ví dụ đề -> ZeroDivisionError: integer division or modulo by zero
đúng: [None, None, None, None, None, None, 20, 40, 0, None, 32]


**Vì sao sai:** sau số 0 mọi `pre` đều bằng 0 ⇒ `0 // 0`. **Sửa:** reset `pre = [1]` khi gặp 0 và trả 0 khi `k ≥ len(pre)`.

### Lỗi 2 — Lệch một: dùng `k > len(pre)` thay vì `k >= len(pre)`

In [22]:
class WrongOffByOne(ProductPrefixReset):
    def getProduct(self, k):
        if k > len(self.pre):                  # nên là >=
            return 0
        idx = max(0, len(self.pre) - 1 - k)    # k == len(pre) bị "kẹp" về phần tử canh
        return self.pre[-1] // self.pre[idx]

print("sai :", run(WrongOffByOne, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, None, None, None, None, None, 20, 40, 40, None, 32]
đúng: [None, None, None, None, None, None, 20, 40, 0, None, 32]


**Vì sao sai:** sau số 0 có `m = len(pre) − 1` phần tử. `k = m + 1 = len(pre)` đã **chạm** số 0 nên tích phải là 0 (ví dụ `getProduct(4)` ra 40 thay vì 0). **Sửa:** điều kiện `k >= len(self.pre)`.

### Lỗi 3 — Reset thành `[]` thay vì `[1]`

In [23]:
class WrongResetEmpty(ProductPrefixReset):
    def add(self, num):
        if num == 0:
            self.pre = []                          # mất phần tử canh 1
        else:
            self.pre.append((self.pre[-1] if self.pre else 1) * num)

print("sai :", run(WrongResetEmpty, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, None, None, None, None, None, 20, 0, 0, None, 32]
đúng: [None, None, None, None, None, None, 20, 40, 0, None, 32]


**Vì sao sai:** mất phần tử canh `1` ⇒ `len(pre)` lệch một so với số phần tử sau 0, và `pre[-k-1]` trỏ sai. **Sửa:** luôn reset thành `[1]`.

### Lỗi 4 — Dùng chia số thực `/`

In [24]:
class WrongFloat(ProductPrefixReset):
    def getProduct(self, k):
        if k >= len(self.pre):
            return 0
        return self.pre[-1] / self.pre[-k - 1]

out = run(WrongFloat, EX_OPS, EX_ARGS)
print("sai :", out, "-> kiểu", type(out[6]).__name__)
print("đúng:", EX_OUT)

sai : [None, None, None, None, None, None, 20.0, 40.0, 0, None, 32.0] -> kiểu float
đúng: [None, None, None, None, None, None, 20, 40, 0, None, 32]


**Vì sao sai:** trả về `float` (`20.0`) thay vì `int`; với tích lớn (ví dụ Cách 4 không reset) phép chia thực còn mất chính xác. **Sửa:** dùng `//`.

### Lỗi 5 — Dùng logarit để "biến tích thành tổng"

In [25]:
import math
class WrongLog:
    def __init__(self):
        self.s = [0.0]
        self.z = [0]
    def add(self, num):
        self.z.append(self.z[-1] + (num == 0))
        self.s.append(self.s[-1] + (math.log(num) if num else 0.0))
    def getProduct(self, k):
        if self.z[-1] - self.z[-k - 1]:
            return 0
        return int(math.exp(self.s[-1] - self.s[-k - 1]))   # cắt phần thập phân

ops = ["P"] + ["add"] * 3 + ["getProduct"]
args = [[]] + [[7], [7], [7]] + [[3]]
print("sai :", run(WrongLog, ops, args), "| đúng:", run(ProductOfNumbers, ops, args))

sai : [None, None, None, None, 342] | đúng: [None, None, None, None, 343]


**Vì sao sai:** `exp(log 343)` ≈ `342.99999999999994` ⇒ `int` cắt thành `342`. Sai số dấu phẩy động! **Sửa:** dùng số nguyên và phép chia nguyên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**k phần tử cuối** của một dòng" + truy vấn tổng/tích/xor ⇒ **tiền tố** (sum/product/xor prefix).
- Tích tiền tố ⇒ luôn hỏi "**còn số 0 thì sao?**" — reset hoặc đếm số 0.
- Phần tử canh (`pre = [1]` / `S = [0]`) giúp công thức không cần nhánh đặc biệt cho `k = m`.
- Bài design: viết bộ `run` + một cài đặt brute force trước để đối chiếu.
- Nhớ đọc đảm bảo của đề ("vừa 32-bit") — giúp biết khi nào reset là đủ để số nhỏ.

## 15. Biến thể của bài

1. **Tổng `k` số cuối** — tiền tố tổng, không cần xử lý 0.
2. **Tích của đoạn bất kỳ `[l, r]`** trong dòng — Cách 4 (đếm số 0 + tích khác 0), xem Bài 5.
3. **Có số âm** — tiền tố tích vẫn đúng (chia hết), chỉ cần xử lý 0.
4. **Tích theo modulo nguyên tố `p`** — thay phép chia bằng nhân **nghịch đảo modulo** (`pow(x, p−2, p)`), cẩn thận các số chia hết cho `p`.
5. **LeetCode 238 — Product of Array Except Self**: tiền tố tích từ trái + hậu tố tích từ phải, không dùng phép chia.

## 16. So sánh các cách

| Cách | Ý tưởng | add | getProduct | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Brute force | nhân `k` số cuối | `O(1)` | `O(k)` | `O(n)` | baseline, trọng tài |
| 1b. Brute + dừng tại 0 | như trên, dừng sớm | `O(1)` | `O(k)` worst | `O(n)` | nhiều số 0 |
| 2. Tiền tố + reset | quên phần trước số 0 | `O(1)` | `O(1)` | `O(n)` | **nộp bài** |
| 3. Tiền tố + last_zero | nhớ vị trí số 0 cuối | `O(1)` | `O(1)` | `O(n)` | cần chỉ số trùng vị trí |
| 4. Đếm 0 + tích khác 0 | hai mảng tiền tố | `O(1)`* | `O(1)`* | `O(n)` | truy vấn đoạn bất kỳ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Chuỗi lệnh `add(2), add(3), add(0), add(4), getProduct(1), getProduct(2)` cho kết quả gì?

**Bài 2 (tính tay):** Viết `pre` (Cách 2) sau mỗi lệnh: `add(1), add(5), add(2), add(0), add(3), add(3)` rồi tính `getProduct(2)`, `getProduct(3)`.

**Bài 3 (điền TODO):** Hoàn thiện lớp `SumOfLastK` (tổng `k` số cuối, `O(1)` mỗi thao tác).

**Bài 4 (tự viết lại):** Viết lại Cách 3 (lưu `last_zero`) mà **không nhìn** code ở trên, kiểm thử bằng bộ ngẫu nhiên.

**Bài 5 (mini challenge):** Lớp `RangeProduct` với `add(num)` và `query(l, r)` trả về tích các phần tử ở vị trí `l..r` (0-indexed, bao gồm hai đầu) của dòng — cả hai thao tác `O(1)`.

In [26]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
class SumOfLastK:
    def __init__(self):
        self.S = [0]
    def add(self, num):
        pass  # TODO: nối tổng tiền tố mới
    def getSum(self, k):
        pass  # TODO: S[-1] - S[-k-1]

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class MyProductLastZero:
    def __init__(self):
        pass  # TODO
    def add(self, num):
        pass  # TODO
    def getProduct(self, k):
        pass  # TODO

class RangeProductTodo:
    def __init__(self):
        pass  # TODO: Z (đếm số 0 tiền tố), Q (tích tiền tố, reset sau số 0?)
    def add(self, num):
        pass  # TODO
    def query(self, l, r):
        pass  # TODO

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Số 0 reset. (2) Sau `add(4)`: `pre = [1, 4]`. (3) `getProduct(2)` có `k = 2 ≥ len(pre)`.

**Bài 2:** (1) Trước số 0: `[1,1,5,10]`. (2) Sau số 0: `[1]`, rồi `[1,3,9]`. (3) `getProduct(3)` có `3 ≥ 3`.

**Bài 3:** (1) `self.S.append(self.S[-1] + num)`. (2) `return self.S[-1] - self.S[-k - 1]`. (3) Không cần xử lý số 0.

**Bài 4:** (1) `pre` không bao giờ bị xóa. (2) Gặp 0: `append(1)` và cập nhật `last_zero`. (3) Đoạn `[n−k+1, n]` chứa 0 ⇔ `last_zero ≥ n−k+1`.

**Bài 5:** (1) `Z[i]` = số số 0 trong `i` phần tử đầu. (2) Để tránh số lớn và chia cho 0, dùng tích "khởi động lại sau mỗi số 0" (giống Cách 3): nếu đoạn không chứa 0 thì nó nằm gọn trong một "khối" giữa hai số 0, khi đó `Q[r+1] // Q[l]` vẫn đúng **chỉ khi** `Q[l]` thuộc cùng khối — điều này đúng nếu ta reset `Q` thành 1 **tại** vị trí số 0. (3) Kiểm thử với brute force `prod(a[l:r+1])`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
ops = ["P", "add", "add", "add", "add", "getProduct", "getProduct"]
args = [[], [2], [3], [0], [4], [1], [2]]
out = run(ProductOfNumbers, ops, args)
print("Bài 1:", out[-2:])
assert out[-2:] == [4, 0]

# Bài 2
ops = ["P"] + ["add"] * 6 + ["getProduct", "getProduct"]
args = [[]] + [[1], [5], [2], [0], [3], [3]] + [[2], [3]]
out = run(ProductTrace, ops, args)
assert out[-2:] == [9, 0]

Bài 1: [4, 0]
add( 1)        pre = [1, 1]
add( 5)        pre = [1, 1, 5]
add( 2)        pre = [1, 1, 5, 10]
add( 0)        pre = [1]  (reset vì gặp 0)
add( 3)        pre = [1, 3]
add( 3)        pre = [1, 3, 9]
getProduct(2): pre[-1]=9 // pre[-3]=1 = 9
getProduct(3): k=3 >= len(pre)=3 -> đoạn chứa 0 -> 0


In [29]:
# Bài 3
class SumOfLastKSol:
    def __init__(self):
        self.S = [0]
    def add(self, num):
        self.S.append(self.S[-1] + num)
    def getSum(self, k):
        return self.S[-1] - self.S[-k - 1]

obj = SumOfLastKSol()
for x in [3, 0, 2, 5, 4]:
    obj.add(x)
print("Bài 3:", obj.getSum(2), obj.getSum(5))
assert obj.getSum(2) == 9 and obj.getSum(5) == 14

# Bài 4 (một lời giải mẫu) + kiểm thử ngẫu nhiên
class MyProductLastZeroSol:
    def __init__(self):
        self.pre, self.z = [1], 0
    def add(self, num):
        if num:
            self.pre.append(self.pre[-1] * num)
        else:
            self.pre.append(1)
            self.z = len(self.pre) - 1
    def getProduct(self, k):
        n = len(self.pre) - 1
        return 0 if self.z > n - k else self.pre[n] // self.pre[n - k]

rng = random.Random(4)
for _ in range(300):
    ops, args = random_ops(rng, 40)
    assert run(MyProductLastZeroSol, ops, args) == run(ProductBrute, ops, args)
print("Bài 4: khớp brute force trên 300 chuỗi lệnh")

Bài 3: 9 14
Bài 4: khớp brute force trên 300 chuỗi lệnh


In [30]:
# Bài 5
class RangeProduct:
    def __init__(self):
        self.a_len = 0
        self.Z = [0]       # Z[i]: số lượng số 0 trong i phần tử đầu
        self.Q = [1]       # Q[i]: tích từ sau số 0 gần nhất (trong i phần tử đầu) đến phần tử i

    def add(self, num):
        self.Z.append(self.Z[-1] + (num == 0))
        self.Q.append(1 if num == 0 else self.Q[-1] * num)

    def query(self, l, r):
        if self.Z[r + 1] - self.Z[l] > 0:
            return 0
        # đoạn [l, r] không có số 0: Q[l] và Q[r+1] cùng thuộc một khối
        return self.Q[r + 1] // self.Q[l]

rng = random.Random(5)
for _ in range(300):
    a = [rng.choice([0, 1, 2, 3, 1, 2]) for _ in range(rng.randint(1, 25))]
    obj = RangeProduct()
    for x in a:
        obj.add(x)
    for _ in range(20):
        l = rng.randrange(len(a)); r = rng.randrange(l, len(a))
        assert obj.query(l, r) == prod(a[l:r + 1]), (a, l, r)
print("Bài 5: RangeProduct khớp prod() trên 300 mảng × 20 truy vấn")

Bài 5: RangeProduct khớp prod() trên 300 mảng × 20 truy vấn


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
class ProductOfNumbers:
    def __init__(self):
        self.pre = [1]
    def add(self, num):
        if num == 0:
            self.pre = [1]                      # reset
        else:
            self.pre.append(self.pre[-1] * num)
    def getProduct(self, k):
        if k >= len(self.pre):
            return 0                            # đoạn chạm số 0
        return self.pre[-1] // self.pre[-k - 1]
```

**Checklist trước khi submit:**
- [ ] Có xử lý số 0 (không bao giờ chia cho 0).
- [ ] Reset thành `[1]` (giữ phần tử canh).
- [ ] Điều kiện trả 0 là `k >= len(pre)` (thử `[3,0,2,5,4]`, `k = 4` ⇒ 0).
- [ ] Dùng `//`, trả về `int`.
- [ ] Không dùng logarit / số thực.
- [ ] Đúng tên lớp/phương thức: `ProductOfNumbers`, `add`, `getProduct`.